# pymupdf4llm — PDF Parse Accuracy Evaluation

**Purpose:** Test `pymupdf4llm` as a replacement for `ai_parse_document` for ingesting procedure PDFs into a RAG pipeline.

**Key things being evaluated:**
- Accuracy of text extraction, especially for multi-page tables
- Determinism (same input → same output every run)
- Detection of cross-page table continuity issues

**Assumptions:** All PDFs are digital (no scanned/OCR PDFs).

## Cell 1 — Install dependencies

In [ ]:
%pip install pymupdf4llm
dbutils.library.restartPython()

## Cell 2 — Imports & config

Update `VOLUME_PATH` and `TEST_PDF` before running.

In [ ]:
import pymupdf4llm
import pymupdf  # bundled with pymupdf4llm
from pathlib import Path
from IPython.display import display, Markdown
import re

# ── Edit these ──────────────────────────────────────────────────────────────
VOLUME_PATH = "/Volumes/<catalog>/<schema>/<volume>"
TEST_PDF    = "your_test_procedure.pdf"
# ────────────────────────────────────────────────────────────────────────────

pdf_path = f"{VOLUME_PATH}/{TEST_PDF}"
print(f"Target PDF: {pdf_path}")

## Cell 3 — Parse PDF with pymupdf4llm

`pymupdf4llm` converts the PDF to Markdown, preserving headers, tables, and lists.  
Using `page_chunks=True` returns one chunk per page, making it easier to inspect and diff individual pages.

In [ ]:
def parse_pdf(path: str) -> list[dict]:
    """
    Parse a PDF and return one dict per page:
      {
        page:       int   (0-based page index),
        text:       str   (Markdown content),
        word_count: int,
        has_tables: bool  (True if markdown table pipes detected)
      }
    """
    chunks = pymupdf4llm.to_markdown(
        path,
        page_chunks=True,    # one chunk per page
        show_progress=False,
    )
    results = []
    for chunk in chunks:
        text = chunk["text"]
        results.append({
            "page":       chunk["metadata"]["page_number"],  # 0-based
            "text":       text,
            "word_count": len(text.split()),
            "has_tables": "|" in text,
        })
    return results


pages = parse_pdf(pdf_path)

print(f"Pages parsed : {len(pages)}")
print(f"Total words  : {sum(p['word_count'] for p in pages):,}")
print(f"Pages w/ tables: {sum(1 for p in pages if p['has_tables'])}")

## Cell 4 — Render a single page as Markdown

The fastest sanity-check: eyeball the rendered output against the actual PDF.  
Change `page_num` to inspect any page (1-based).

In [ ]:
def show_page(pages: list[dict], page_num: int) -> None:
    """Render a 1-based page number as formatted Markdown in the notebook."""
    match = next((p for p in pages if p["page"] == page_num - 1), None)
    if not match:
        print(f"Page {page_num} not found (doc has {len(pages)} pages).")
        return
    print(
        f"── Page {page_num}  |  {match['word_count']} words  "
        f"|  tables detected: {match['has_tables']} ──\n"
    )
    display(Markdown(match["text"]))


# ── Change this to inspect a different page ──────────────────────────────────
show_page(pages, 1)
# ─────────────────────────────────────────────────────────────────────────────

## Cell 5 — Extract all tables from parsed Markdown

Isolates every table on every page so you can verify row completeness.  
Cross-page table splits (your main pain point with `ai_parse_document`) are flagged in Cell 7.

In [ ]:
def extract_tables_from_markdown(pages: list[dict]) -> list[dict]:
    """
    Pull out raw markdown table blocks with their source page.
    A markdown table is a contiguous block of lines containing '|'.
    Returns list of dicts: {page, row_count, raw}
    """
    # Matches a markdown table separator row like |---|---|---|
    separator_re = re.compile(r"^\|[-| :]+\|$")
    tables = []

    for p in pages:
        lines     = p["text"].splitlines()
        in_table  = False
        table_buf = []

        for line in lines:
            if "|" in line:
                in_table = True
                table_buf.append(line)
            else:
                if in_table and table_buf:
                    data_rows = sum(
                        1 for l in table_buf
                        if "|" in l and not separator_re.match(l.strip())
                    )
                    tables.append({
                        "page":      p["page"] + 1,  # convert to 1-based
                        "row_count": data_rows,
                        "raw":       "\n".join(table_buf),
                    })
                    table_buf = []
                in_table = False

        # Flush if a table runs right to the end of the page
        if table_buf:
            data_rows = sum(
                1 for l in table_buf
                if "|" in l and not separator_re.match(l.strip())
            )
            tables.append({
                "page":      p["page"] + 1,
                "row_count": data_rows,
                "raw":       "\n".join(table_buf),
            })

    return tables


tables = extract_tables_from_markdown(pages)

print(f"Tables found: {len(tables)}")
for i, t in enumerate(tables, 1):
    print(f"  Table {i}: page {t['page']}, {t['row_count']} data rows")

## Cell 6 — Render a specific table

Use this to manually verify row completeness against the source PDF.  
Change `table_num` to inspect a different table.

In [ ]:
def show_table(tables: list[dict], table_num: int) -> None:
    if table_num < 1 or table_num > len(tables):
        print(f"Table {table_num} not found (doc has {len(tables)} tables).")
        return
    t = tables[table_num - 1]
    print(f"Table {table_num} — page {t['page']} — {t['row_count']} data rows\n")
    display(Markdown(t["raw"]))


# ── Change this to inspect a different table ─────────────────────────────────
show_table(tables, 1)
# ─────────────────────────────────────────────────────────────────────────────

## Cell 7 — Cross-page table continuity check

Flags pairs of tables on consecutive pages that are likely one logical table split by a page break.  
These are exactly the rows `ai_parse_document` was dropping — verify that both the last row of the
first fragment and the first row of the second fragment are intact and make sense together.

In [ ]:
def flag_cross_page_tables(tables: list[dict]) -> None:
    """
    Heuristic: two tables on consecutive pages are likely one logical table
    split by a page break. Print the boundary rows for manual review.
    """
    found_any = False
    for i in range(len(tables) - 1):
        a, b = tables[i], tables[i + 1]
        if b["page"] == a["page"] + 1:
            found_any = True
            print(
                f"⚠️  Possible cross-page table: "
                f"Table {i+1} (p{a['page']}) → Table {i+2} (p{b['page']})"
            )
            last_row  = a["raw"].splitlines()[-1][:120]
            first_row = b["raw"].splitlines()[0][:120]
            print(f"   Last row  of Table {i+1} ➜ {last_row}")
            print(f"   First row of Table {i+2} ➜ {first_row}")
            print()

    if not found_any:
        print("✅  No cross-page tables detected.")


flag_cross_page_tables(tables)

## Cell 8 — Determinism verification

Parses the same PDF multiple times and confirms the output is identical every run.  
This directly addresses the non-determinism issue observed with `ai_parse_document`.

In [ ]:
def verify_determinism(path: str, runs: int = 3) -> None:
    print(f"Parsing '{Path(path).name}' {runs} times...")
    outputs = [
        pymupdf4llm.to_markdown(path, show_progress=False)
        for _ in range(runs)
    ]
    all_equal = all(o == outputs[0] for o in outputs[1:])
    if all_equal:
        print(f"✅  Output is identical across all {runs} runs.")
    else:
        for i, o in enumerate(outputs[1:], 2):
            if o != outputs[0]:
                # Find the first character position that differs
                diff_pos = next(
                    (j for j, (c1, c2) in enumerate(zip(outputs[0], o)) if c1 != c2),
                    min(len(outputs[0]), len(o))
                )
                print(f"⚠️  Run {i} differs from Run 1 at character position {diff_pos}.")
                print(f"   Run 1 excerpt: ...{outputs[0][max(0,diff_pos-30):diff_pos+30]}...")
                print(f"   Run {i} excerpt: ...{o[max(0,diff_pos-30):diff_pos+30]}...")


verify_determinism(pdf_path, runs=3)

## Cell 9 — Evaluation summary

Quick scorecard across all evaluation checks.

In [ ]:
def evaluation_summary(pages: list[dict], tables: list[dict]) -> None:
    total_words   = sum(p["word_count"] for p in pages)
    empty_pages   = [p["page"] + 1 for p in pages if p["word_count"] == 0]
    cross_page_ct = sum(
        1 for i in range(len(tables) - 1)
        if tables[i + 1]["page"] == tables[i]["page"] + 1
    )

    print("══ Parse Evaluation Summary ══════════════════════════════")
    print(f"  PDF              : {Path(pdf_path).name}")
    print(f"  Pages parsed     : {len(pages)}")
    print(f"  Total word count : {total_words:,}")
    print(f"  Empty pages      : {empty_pages if empty_pages else 'none'}")
    print(f"  Tables detected  : {len(tables)}")
    print(f"  Cross-page tables: {cross_page_ct}  ← manual review needed")
    print("══════════════════════════════════════════════════════════")
    print()
    print("Manual review checklist:")
    checks = [
        ("Table row completeness",      "Cell 6 — render each table, count rows against the PDF"),
        ("Cross-page table continuity", "Cell 7 — confirm last/first boundary rows stitch correctly"),
        ("Header hierarchy",            "Cell 4 — #/##/### should mirror the PDF heading levels"),
        ("Determinism",                 "Cell 8 — should always print True"),
        ("Empty pages",                 "Summary above — investigate any page with real content showing 0 words"),
    ]
    for label, guidance in checks:
        print(f"  [ ] {label:<30} — {guidance}")


evaluation_summary(pages, tables)

## Cell 10 — Pre-processing & section-level DataFrame

Produces one row per **section** rather than one row per page. Sections that span a page break
are merged into a single row so the RAG retriever sees complete, coherent content.

**DataFrame schema:**

| Column | Description |
|---|---|
| `doc_title` | Document ID extracted from first page (e.g. `EM-1.3`) |
| `category` | Value after `Category:` on first page |
| `topic` | Value after `Topic:` on first page |
| `published` | Value after `Published:` on first page |
| `section_header` | Text of the `## **Header**` that opens the section; `None` for preamble content before the first header |
| `start_page` | First page (0-based) on which this section's content appears |
| `end_page` | Last page (0-based) on which this section's content appears |
| `text` | Cleaned Markdown text for the full section |
| `word_count` | Word count of the section text |

**Cleaning applied to every page before splitting:**
1. Image placeholders — `picture [NNN x NNN] intentionally omitted`
2. Page footer — `FCA Examination Manual Financial Institution Rating System`
3. Page number lines — `Page N`
4. Standalone attachment headers — `Attachment N`
5. Table rows containing only `Attachment N`

In [ ]:
import pandas as pd

# ── Regex patterns ────────────────────────────────────────────────────────────

# --- Cleaning ---
# Image placeholder: e.g. 'picture [467 x 59] intentionally omitted'
RE_IMAGE = re.compile(
    r"picture\s*\[\d+\s*x\s*\d+\][^\n]*\n?", re.IGNORECASE
)
# Page footer — \w+ after 'Examinat' tolerates the known typo 'Examinatinon'
RE_FOOTER = re.compile(
    r"FCA\s+Examinat\w+\s+Manual\s+Financial\s+Institution\s+Rating\s+System[^\n]*\n?",
    re.IGNORECASE,
)
# Page number line — must be the entire line to avoid clipping mid-sentence 'Page'
RE_PAGE_NUM = re.compile(r"^Page\s+\d+\s*$", re.MULTILINE | re.IGNORECASE)
# Standalone 'Attachment N' line
RE_ATTACHMENT = re.compile(r"^Attachment\s+\d+\s*$", re.MULTILINE | re.IGNORECASE)
# Markdown table row whose only cell content is 'Attachment N' (optional bold markers)
RE_TABLE_ATT = re.compile(
    r"^\|[\s*]*Attachment\s+\d+[\s*]*\|.*$", re.MULTILINE | re.IGNORECASE
)

# --- Metadata extraction ---
# Document title: e.g. 'EM-1.3' — up to 6 uppercase letters, hyphen, digits and dots
RE_TITLE = re.compile(r"^([A-Z]{1,6}-[\d.]+)\b", re.MULTILINE)
RE_CAT   = re.compile(r"Category:\s*(.+)")
RE_TOPIC = re.compile(r"Topic:\s*(.+)")
RE_PUB   = re.compile(r"Published:\s*(.+)")

# --- Section splitting ---
# Splitting pattern: captures the full '## **Header**' token so it stays in the list
RE_HEADER_SPLIT = re.compile(r"(##\s+\*\*.*?\*\*)", re.MULTILINE)
# Extracts the header label from inside the bold markers
RE_HEADER_TEXT  = re.compile(r"\*\*(.+?)\*\*")


# ── Helper functions ──────────────────────────────────────────────────────────

def extract_first_page_metadata(text: str) -> dict:
    """
    Extract doc-level metadata from the first page's raw text.
    Returns dict with keys: doc_title, category, topic, published.
    Values are None when the pattern is not found.
    """
    def first(pattern, src):
        m = pattern.search(src)
        return m.group(1).strip() if m else None

    return {
        "doc_title": first(RE_TITLE, text),
        "category":  first(RE_CAT,   text),
        "topic":     first(RE_TOPIC,  text),
        "published": first(RE_PUB,    text),
    }


def clean_text(text: str) -> str:
    """
    Remove boilerplate from a single page's Markdown text, then collapse
    any runs of 3+ blank lines left behind into a single blank line.
    """
    text = RE_IMAGE.sub("", text)
    text = RE_FOOTER.sub("", text)
    text = RE_PAGE_NUM.sub("", text)
    text = RE_ATTACHMENT.sub("", text)
    text = RE_TABLE_ATT.sub("", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


# ── Section-level DataFrame builder ──────────────────────────────────────────

def build_section_dataframe(pages: list[dict]) -> pd.DataFrame:
    """
    Convert a list of page dicts (from parse_pdf) into a section-level DataFrame.

    Logic:
      - Doc-level metadata (title, category, topic, published) is extracted
        from the first page and stamped on every row.
      - Each page's cleaned text is split on '## **Header**' markers.
      - Text appearing before the first header (cover page, preamble) becomes
        a row with section_header = None.
      - When a section continues onto the next page (no new header opens),
        the text is appended to the current section's accumulator rather than
        starting a new row — so cross-page sections stay together.
      - start_page / end_page track which pages contribute to each row.
    """
    doc_meta = extract_first_page_metadata(pages[0]["text"])

    # ── Accumulators for the section currently being built ────────────────────
    current_header = None
    current_parts  = []       # list of text fragments collected so far
    current_start  = pages[0]["page"]
    current_end    = pages[0]["page"]

    records = []

    def flush() -> None:
        """Persist the current accumulated section as a record."""
        text = "\n\n".join(current_parts).strip()
        if not text:
            return
        records.append({
            **doc_meta,
            "section_header": current_header,
            "start_page":     current_start,
            "end_page":       current_end,
            "text":           text,
            "word_count":     len(text.split()),
        })

    for p in pages:
        cleaned  = clean_text(p["text"])
        page_num = p["page"]

        # re.split with a capturing group keeps the matched header tokens
        # in the result list, interleaved with the surrounding text:
        #   [pre_text, header1, text1, header2, text2, ...]
        parts = RE_HEADER_SPLIT.split(cleaned)

        for part in parts:
            if RE_HEADER_SPLIT.match(part):
                # ── Section header token ─────────────────────────────────────
                # Save whatever was accumulating before this header
                flush()
                # Start a fresh accumulator for the new section
                current_header = RE_HEADER_TEXT.search(part).group(1).strip()
                current_parts  = []
                current_start  = page_num
                current_end    = page_num
            else:
                # ── Body text token ──────────────────────────────────────────
                stripped = part.strip()
                if stripped:
                    current_parts.append(stripped)
                    current_end = page_num   # extend end_page as content grows

    # Flush the final section after all pages are processed
    flush()

    return pd.DataFrame(records, columns=[
        "doc_title", "category", "topic", "published",
        "section_header", "start_page", "end_page",
        "text", "word_count",
    ])


# ── Build & inspect ───────────────────────────────────────────────────────────
df = build_section_dataframe(pages)

print(f"Sections extracted : {len(df)}")
print(f"Total words        : {df['word_count'].sum():,}")
print(f"Cross-page sections: {len(df[df['start_page'] != df['end_page']])}")
print()
print("Schema:")
print(df.dtypes)
print()
print("Section overview (header + page range + word count):")
display(df[["section_header", "start_page", "end_page", "word_count"]])
print()
print("Doc-level metadata (should be identical on every row):")
display(df[["doc_title", "category", "topic", "published"]].drop_duplicates())

## Cell 11 — Spot-check a section

Render a specific section's text as Markdown to verify the content and section boundary look correct.  
Change `section_num` to inspect a different row (1-based).

In [ ]:
def show_section(df: pd.DataFrame, section_num: int) -> None:
    """Render a section row (1-based) as formatted Markdown."""
    if section_num < 1 or section_num > len(df):
        print(f"Section {section_num} not found (DataFrame has {len(df)} rows).")
        return
    row = df.iloc[section_num - 1]
    header = row['section_header'] or '(preamble — no section header)'
    pages  = (
        f"p{row['start_page']}"
        if row['start_page'] == row['end_page']
        else f"pp{row['start_page']}–{row['end_page']}"
    )
    print(f"── Section {section_num}: '{header}'  |  {pages}  |  {row['word_count']} words ──\n")
    display(Markdown(row["text"]))


# ── Change this to inspect a different section ────────────────────────────────
show_section(df, 1)
# ─────────────────────────────────────────────────────────────────────────────